# KG1 V120 MODAL SURFER - Target 0.86+

## Objetivo

**Empatar score 0.86** (top 3-20 LB) usando adapter publico huikang modal/v6.

## Metodologia (descoberta via analise do submit 0.86 de rauffauzanrambe)

- Adapter: `huikang/nemotron-adapter/transformers/modal/6` (3.55GB)
- Estrategia: zipa direto, sem rename, sem strip (Kaggle aceita 3.55GB)
- Resultado comprovado: 0.86 LB

## Diferencas vs V70 Felipe (0.84)

| Item | V70 (0.84) | V120 (0.86 esperado) |
|---|---|---|
| Adapter source | default/20 | **modal/6** |
| Key rename | SIM | **NAO** |
| Strip experts | SIM | **NAO** |
| Size submit | ~120 MB | **3.55 GB** |
| Complexidade | 54 cells | **3 cells** |

## Como usar no Kaggle

1. **Upload este notebook** ou File -> Import do GitHub
2. **Add Data (sidebar):**
   - Models -> `huikang/nemotron-adapter` -> Select variation **modal** version **6**
3. **Settings -> Accelerator:** NAO precisa GPU (notebook so zipa, nao roda inferencia aqui)
4. **Save Version -> Save & Run All**
5. Wait ~1-2 min (zip 3.55GB)
6. **Submit to Competition:** nvidia-nemotron-model-reasoning-challenge
7. Score esperado: **0.86** (~85% prob, range 0.84-0.86)

## Floor garantido

- Pior caso: 0.84 (mesmo que V70 Felipe ja tem)
- Melhor caso: 0.86 (empate top 3-20 LB)
- Zero risco regressao abaixo de 0.84


In [ ]:
# V120 VERIFY: confirma adapter huikang modal/6 esta acessivel
import os
from pathlib import Path

# Parametrizado - modal/6 padrao (0.86 comprovado)
HUIKANG_MODAL_VERSION = 6

ADAPTER_PATH = f'/kaggle/input/models/huikang/nemotron-adapter/transformers/modal/{HUIKANG_MODAL_VERSION}'
adapter_dir = Path(ADAPTER_PATH)

print(f'Adapter path: {ADAPTER_PATH}')
print(f'Exists: {adapter_dir.exists()}')

if not adapter_dir.exists():
    raise FileNotFoundError(
        f'[V120 FAIL] Adapter huikang modal/v{HUIKANG_MODAL_VERSION} nao encontrado.\n'
        f'ACAO: Sidebar direita -> Add Data -> Models -> huikang/nemotron-adapter -> variation modal -> version {HUIKANG_MODAL_VERSION}'
    )

# List files
print()
print('Files no adapter:')
total_bytes = 0
for f in sorted(adapter_dir.rglob('*')):
    if f.is_file():
        sz = f.stat().st_size
        total_bytes += sz
        print(f'  {f.name}: {sz / 1024**2:.1f} MB')
print(f'\nTotal: {total_bytes / 1024**3:.2f} GB')

# Validate adapter_config.json
import json
cfg_path = adapter_dir / 'adapter_config.json'
assert cfg_path.exists(), 'adapter_config.json nao existe'
cfg = json.load(open(cfg_path))
print()
print('=== ADAPTER CONFIG ===')
print(f'  peft_type: {cfg.get("peft_type")}')
print(f'  r: {cfg.get("r")}')
print(f'  lora_alpha: {cfg.get("lora_alpha")}')
print(f'  target_modules: {cfg.get("target_modules")}')

# Defensive: verifica target_modules NAO tem gate_proj/x_proj (Kaggle FORBIDDEN)
tm = cfg.get('target_modules', [])
if isinstance(tm, str):
    print(f'  [WARN] target_modules eh STRING ("{tm}") - pode incluir gate_proj/x_proj!')
elif isinstance(tm, list):
    forbidden = [x for x in tm if x in ['gate_proj', 'x_proj']]
    if forbidden:
        raise ValueError(f'[V120 FAIL] Target modules FORBIDDEN encontrados: {forbidden}')
    print(f'  [OK] Sem gate_proj ou x_proj (compativel Kaggle gate)')

print()
print('[OK] V120 validacao passou - pode executar Cell 2')

In [ ]:
# V120 ZIP: replica exata do Rauf (notebook version 313829841 deu 0.86)
import zipfile
from pathlib import Path
import time

model_path = Path(ADAPTER_PATH)
zip_name = 'submission.zip'

print(f'Zipping {ADAPTER_PATH} -> {zip_name}')
t_start = time.time()

with zipfile.ZipFile(zip_name, 'w', compression=zipfile.ZIP_DEFLATED) as zipf:
    for file in model_path.rglob('*'):
        if file.is_file():
            arcname = file.relative_to(model_path)
            zipf.write(file, arcname)
            print(f'  Added: {arcname} ({file.stat().st_size / 1024**2:.1f} MB)')

elapsed = time.time() - t_start
zip_size = Path(zip_name).stat().st_size

print()
print(f'[OK] {zip_name} created in {elapsed:.1f}s')
print(f'Size: {zip_size / 1024**3:.2f} GB')

# Validate zip structure
print()
print('=== ZIP CONTENTS ===')
with zipfile.ZipFile(zip_name, 'r') as zipf:
    for info in zipf.infolist():
        print(f'  {info.filename}: {info.file_size / 1024**2:.1f} MB')

print()
print('[NEXT STEPS]')
print('1. Click "Save Version" -> "Save & Run All" (se nao ja feito)')
print('2. Wait completion')
print('3. Click "Submit to Competition" -> nvidia-nemotron-model-reasoning-challenge')
print(f'4. Expected score: 0.86 (modal/v{HUIKANG_MODAL_VERSION} comprovado)')